In [94]:
# Import libraries
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from pathlib import Path
import pickle

In [95]:
# Load dataset
df = pd.read_csv("../Data/raw/Students_Performance_dataset.csv")

In [96]:
# Build student profiles
def build_profile(row):
    return f"Skills: {row['What are the skills do you have ?']}, " \
           f"Interests: {row['What is you interested area?']}, " \
           f"CGPA: {row['What is your current CGPA?']}, " \
           f"Family Income: {row['What is your monthly family income?']}"

df["Profile_Text"] = df.apply(build_profile, axis=1)

In [97]:
# Load embeddings instead of recomputing
model_path = Path("../Models/sbert")
try:
    with open(model_path / "profile_embeddings.pkl", "rb") as f:
        profile_embeddings = pickle.load(f)

    with open(model_path / "scholarship_embeddings.pkl", "rb") as f:
        scholarship_embeddings = pickle.load(f)

    with open(model_path / "scholarship_list.pkl", "rb") as f:
        scholarships = pickle.load(f)

    print("Embeddings loaded successfully!")
except FileNotFoundError:
    print("Embeddings not found. Please run Word_Embedding/04_Embeddings.ipynb first.")

Embeddings loaded successfully!


In [98]:
# Compute similarity matrix
similarity_scores = util.cos_sim(profile_embeddings, scholarship_embeddings)

In [99]:
# Rank scholarships with eligibility filters
filtered_recommendations = []
for student_idx, profile in enumerate(df["Profile_Text"].tolist()):
    scores = similarity_scores[student_idx].cpu().tolist()
    ranked = sorted(list(enumerate(scores)), key=lambda x: x[1], reverse=True)

    # Extract student info directly from df
    cgpa = float(df.iloc[student_idx]["What is your current CGPA?"])
    income = int(df.iloc[student_idx]["What is your monthly family income?"])

    top_matches = []
    for i, score in ranked[:3]:
        sch = scholarships[i]

        # Example eligibility rules
        if "CGPA above 3.5" in sch and cgpa < 3.5:
            continue
        if "income below 30,000" in sch and income > 30000:
            continue

        top_matches.append((sch, score))

    if top_matches:  # only add if at least one scholarship passes filters
        filtered_recommendations.append({
            "Student": profile,
            "Top_Scholarships": top_matches
        })


In [100]:
# Convert recommendations into DataFrame
rows = []
for rec in filtered_recommendations:
    student = rec["Student"]
    for sch, score in rec["Top_Scholarships"]:
        rows.append({
            "Student_Profile": student,
            "Scholarship": sch,
            "Match_Score": round(score, 4)
        })
recommendations_df = pd.DataFrame(rows)

In [101]:
# Save to CSV
output_path = Path("../Data/processed/scholarship_recommendations.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
recommendations_df.to_csv(output_path, index=False)

print(f"Exported recommendations to {output_path}")

Exported recommendations to ..\Data\processed\scholarship_recommendations.csv


In [102]:
# Convert recommendations into DataFrame
rows = []
for rec in filtered_recommendations:
    student = rec["Student"]
    for sch, score in rec["Top_Scholarships"]:
        rows.append({
            "Student_Profile": student,
            "Scholarship": sch,
            "Match_Score": round(score, 4)
        })

recommendations_df = pd.DataFrame(rows)

In [103]:
# Save to CSV
output_path = Path("../Data/processed/scholarship_recommendations.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
recommendations_df.to_csv(output_path, index=False)

print("Exported recommendations to ../Data/processed/scholarship_recommendations.csv")

# Step 12: Preview the first few rows
display(recommendations_df.head())

Exported recommendations to ../Data/processed/scholarship_recommendations.csv


,Student_Profile,Scholarship,Match_Score
0,"Skills: Software Development, Interests: Data ...",Networking and Cybersecurity Scholarship for s...,0.3123
1,"Skills: Software Development, Interests: Data ...",Need-based scholarship for students with famil...,0.2104
2,"Skills: Web development, Interests: Event mana...",Networking and Cybersecurity Scholarship for s...,0.2934
3,"Skills: Programming, Interests: Software, CGPA...",Networking and Cybersecurity Scholarship for s...,0.3298
4,"Skills: Programming, Interests: Artificial Int...",Networking and Cybersecurity Scholarship for s...,0.3157
